# ClinicalDistill — Qwen 1.5 1.8B QLoRA (Kaggle P100)
## Experiment 6

Same model as Experiment 5 (Qwen1.5-1.8B) but with 4-bit quantization.
Lets us compare LoRA vs QLoRA on the same model — same tradeoff analysis as Gemma.

| Model             | Params | Valid JSON | F1    | Urgent Acc |
|-------------------|--------|------------|-------|------------|
| Gemma-3-1B LoRA   | 1B     | 100%       | 0.781 | 85.7%      |
| Gemma-3-1B QLoRA  | 1B     | 100%       | 0.740 | 82.9%      |
| Qwen1.5-1.8B LoRA | 1.8B   | ___        | ___   | ___        |
| Qwen1.5-1.8B QLoRA| 1.8B   | ___        | ___   | ___        |

**Before running:** Settings → Internet → On

In [ ]:
!pip install -q bitsandbytes transformers peft accelerate datasets trl

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os

DATA_DIR = "/kaggle/input/datasets/janushishastri/clinicalextractdataset"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

!ls {DATA_DIR}
!wc -l {DATA_DIR}/train_fixed.jsonl {DATA_DIR}/test_fixed.jsonl

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
from datasets import load_dataset
import json

dataset = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train_fixed.jsonl",
    "test":  f"{DATA_DIR}/test_fixed.jsonl"
})

print(dataset)
print(dataset["train"][0])

In [ ]:
def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)
    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen1.5-1.8B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model: {model_id}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

args = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/qwen-qlora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

start_time = time.time()
print("Starting Qwen1.5 QLoRA training...")
trainer.train()
training_time = time.time() - start_time

print(f"Training complete: {training_time:.0f}s ({training_time/60:.1f} min)")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
SAVE_PATH = f"{OUTPUT_DIR}/qwen-qlora-final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to: {SAVE_PATH}")

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

SAVE_PATH = f"{OUTPUT_DIR}/qwen-qlora-final"

tokenizer_loaded = AutoTokenizer.from_pretrained(SAVE_PATH)
tokenizer_loaded.pad_token = tokenizer_loaded.eos_token
tokenizer_loaded.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen1.5-1.8B-Chat",
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer_loaded.pad_token_id
model_loaded = PeftModel.from_pretrained(base_model, SAVE_PATH)

print("Model loaded for inference")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def extract_clinical(text):
    prompt = f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{text}</input>
<output>"""

    inputs = tokenizer_loaded(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_loaded.eos_token_id
        )
    response = tokenizer_loaded.decode(outputs[0], skip_special_tokens=True)
    return response.split("<output>")[-1].strip()


test_cases = [
    "Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.",
    "Child presents with mild fever since yesterday, runny nose and occasional cough.",
    "Patient reports sharp lower back pain for a week, worse when sitting, no fever."
]
for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

In [ ]:
import json

def parse_output(text):
    try:
        text = text.strip().replace("</output>", "").strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))
    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

valid_json_count = 0
f1_scores = []
urgent_correct = 0

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]
    if is_valid:
        valid_json_count += 1
        f1_scores.append(symptom_f1(pred.get("symptoms", []), true.get("symptoms", [])))
        if pred.get("urgent") == true.get("urgent"):
            urgent_correct += 1

total = len(dataset["test"])
avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — Qwen1.5-1.8B QLoRA")
print("=" * 50)
print(f"Valid JSON rate : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1  : {avg_f1:.3f}")
print(f"Urgent Accuracy : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

## Results — Copy into Research Log

| Metric           | Gemma LoRA | Gemma QLoRA | Qwen LoRA | Qwen QLoRA |
|------------------|------------|-------------|-----------|------------|
| Valid JSON       | 100%       | 100%        | ___       | ___        |
| Symptom F1       | 0.781      | 0.740       | ___       | ___        |
| Urgent Accuracy  | 85.7%      | 82.9%       | ___       | ___        |
| Params           | 1B         | 1B          | 1.8B      | 1.8B       |
| Training time    | ~2 min     | ~4 min      | ___       | ___        |